In [23]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [4]:
# Upload the padding data

X_train_padded = np.load(
    "../Dataset/processed/X_train_padded.npy"
)

X_val_padded = np.load(
    "../Dataset/processed/X_val_padded.npy"
)

X_test_padded = np.load(
    "../Dataset/processed/X_test_padded.npy"
)

y_train = np.load(
    "../Dataset/processed/y_train.npy"
)

y_val = np.load(
    "../Dataset/processed/y_val.npy"
)

y_test = np.load(
    "../Dataset/processed/y_test.npy"
)

In [5]:
print("X_train:", X_train_padded.shape)
print("X_val:", X_val_padded.shape)
print("X_test:", X_test_padded.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (34705, 200)
X_val: (7439, 200)
X_test: (7438, 200)
y_train: (34705,)
y_val: (7439,)
y_test: (7438,)


In [7]:
with open(
    "../Dataset/processed/feature_config.pkl",
    "rb"
) as file:

    feature_config = pickle.load(file)

In [8]:
VOCAB_SIZE = feature_config["vocab_size"]
EMBEDDING_DIM = feature_config["embedding_dim"]
MAX_SEQUENCE_LENGTH = feature_config["max_sequence_length"]

print("Vocabulary size:", VOCAB_SIZE)
print("Embedding dimension:", EMBEDDING_DIM)
print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)

Vocabulary size: 20000
Embedding dimension: 128
Maximum sequence length: 200


In [9]:
# Load TF-IDF Features for ANN

with open(
    "../Dataset/processed/tfidf_vectorizer.pkl",
    "rb"
) as file:

    tfidf_vectorizer = pickle.load(file)

In [10]:
with open(
    "../Dataset/processed/text_splits.pkl",
    "rb"
) as file:

    text_splits = pickle.load(file)

In [11]:
X_train_text = text_splits["X_train"]
X_val_text = text_splits["X_val"]
X_test_text = text_splits["X_test"]

In [12]:
# Fit already fitted TF-IDF vectorizer

X_train_tfidf = tfidf_vectorizer.transform(
    X_train_text
)

X_val_tfidf = tfidf_vectorizer.transform(
    X_val_text
)

X_test_tfidf = tfidf_vectorizer.transform(
    X_test_text
)

In [13]:
print("TF-IDF train:", X_train_tfidf.shape)
print("TF-IDF validation:", X_val_tfidf.shape)
print("TF-IDF test:", X_test_tfidf.shape)

TF-IDF train: (34705, 20000)
TF-IDF validation: (7439, 20000)
TF-IDF test: (7438, 20000)


In [14]:
# Basic ANN

TFIDF_FEATURES = X_train_tfidf.shape[1]

print("TF-IDF features:", TFIDF_FEATURES)

TF-IDF features: 20000


In [15]:
ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dense(
        32,
        activation="relu"
    ),
    
    
    Dense(
        1,
        activation="sigmoid"
    )
])

E0000 00:00:1787495382.265197  175934 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [16]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [17]:
ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │     1,280,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,282,177 (4.89 MB)

 Trainable params: 1,282,177 (4.89 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64,
    
    verbose=1
)

W0000 00:00:1787495458.050439  175934 cpu_allocator_impl.cc:82] Allocation of 57729328 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.8785 - loss: 0.3091 - val_accuracy: 0.9016 - val_loss: 0.2404
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 22s 29ms/step - accuracy: 0.9545 - loss: 0.1327 - val_accuracy: 0.8926 - val_loss: 0.2822
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9780 - loss: 0.0714 - val_accuracy: 0.8835 - val_loss: 0.3599
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9882 - loss: 0.0367 - val_accuracy: 0.8820 - val_loss: 0.4603
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9952 - loss: 0.0156 - val_accuracy: 0.8771 - val_loss: 0.5951
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9983 - loss: 0.0059 - val_accuracy: 0.8801 - val_loss: 0.7137
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9995 - loss: 0.0028 - val_accuracy: 0.8775 - val_loss: 0.7740
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9998 - loss: 0.0011 - 

In [26]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


In [29]:
ann_probabilities

array([[1.       ],
       [0.8260797],
       [0.9999952],
       ...,
       [0.9999986],
       [0.9999991],
       [1.       ]], shape=(7438, 1), dtype=float32)

In [28]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [30]:
ann_predictions

array([1, 1, 1, ..., 1, 1, 1], shape=(7438,))

In [31]:
def evaluate_model(y_true, y_pred, y_prob):
    
    accuracy = accuracy_score(
        y_true,
        y_pred
    )
    
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )
    
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )
    
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )
    
    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )
    
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

In [32]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [33]:
ann_metrics

{'Accuracy': 0.8744286098413552,
 'Precision': 0.8765133171912833,
 'Recall': 0.8727564961157246,
 'F1 Score': 0.8746308724832215,
 'ROC-AUC': 0.9483536883173129}

In [34]:
ann_model.save(
    "../models/ann/ann_baseline.keras"
)

In [35]:
with open(
    "../models/ann/ann_history.pkl",
    "wb"
) as file:

    pickle.dump(
        ann_history.history,
        file
    )

In [36]:
# Simple RNN

rnn_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    SimpleRNN(
        64
    ),
    
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [37]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [38]:
rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,574,465 (9.82 MB)

 Trainable params: 2,574,465 (9.82 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 64s 110ms/step - accuracy: 0.5030 - loss: 0.6959 - val_accuracy: 0.4999 - val_loss: 0.6946
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 81s 149ms/step - accuracy: 0.5463 - loss: 0.6599 - val_accuracy: 0.4976 - val_loss: 0.7276
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 63s 116ms/step - accuracy: 0.5789 - loss: 0.5989 - val_accuracy: 0.5096 - val_loss: 0.7684
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 60s 111ms/step - accuracy: 0.5714 - loss: 0.6179 - val_accuracy: 0.4981 - val_loss: 0.7839
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 67s 124ms/step - accuracy: 0.5819 - loss: 0.5922 - val_accuracy: 0.4982 - val_loss: 0.8063
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 66s 121ms/step - accuracy: 0.5841 - loss: 0.5849 - val_accuracy: 0.4943 - val_loss: 0.7976
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 63s 115ms/step - accuracy: 0.5864 - loss: 0.5935 - val_accuracy: 0.5071 - val_loss: 0.8274
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 79s 146ms/step - accuracy: 0.5873 - loss: 0

In [40]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step


In [41]:
rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

In [42]:
rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

In [43]:
rnn_metrics

{'Accuracy': 0.4977144393654208,
 'Precision': 0.4997791844545856,
 'Recall': 0.9094562014465577,
 'F1 Score': 0.6450693520805624,
 'ROC-AUC': 0.50002244995125}

In [44]:
rnn_model.save(
    "../models/rnn/rnn_baseline.keras"
)

In [45]:
with open(
    "../models/rnn/rnn_history.pkl",
    "wb"
) as file:

    pickle.dump(
        rnn_history.history,
        file
    )

In [46]:
# LSTM

lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    LSTM(
        64
    ),
    
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [47]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [48]:
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,611,521 (9.96 MB)

 Trainable params: 2,611,521 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 171s 308ms/step - accuracy: 0.5355 - loss: 0.6830 - val_accuracy: 0.5382 - val_loss: 0.6797
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 143s 263ms/step - accuracy: 0.5551 - loss: 0.6630 - val_accuracy: 0.5314 - val_loss: 0.6790
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 145s 267ms/step - accuracy: 0.6301 - loss: 0.6058 - val_accuracy: 0.5407 - val_loss: 0.6749
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 143s 263ms/step - accuracy: 0.8535 - loss: 0.3299 - val_accuracy: 0.8847 - val_loss: 0.2958
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 100s 184ms/step - accuracy: 0.9380 - loss: 0.1796 - val_accuracy: 0.8840 - val_loss: 0.3106
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 109s 200ms/step - accuracy: 0.9643 - loss: 0.1176 - val_accuracy: 0.8787 - val_loss: 0.3556
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 108s 199ms/step - accuracy: 0.9812 - loss: 0.0718 - val_accuracy: 0.8758 - val_loss: 0.4252
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 107s 198ms/step - accuracy: 0.9898 -

In [51]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step


In [52]:
lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

In [53]:
lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

In [54]:
lstm_metrics

{'Accuracy': 0.8652863673030384,
 'Precision': 0.8796219071448429,
 'Recall': 0.8475756763996786,
 'F1 Score': 0.8633015006821282,
 'ROC-AUC': 0.9320514085807979}

In [55]:
lstm_model.save(
    "../models/lstm/lstm_baseline.keras"
)

In [56]:
with open(
    "../models/lstm/lstm_history.pkl",
    "wb"
) as file:

    pickle.dump(
        lstm_history.history,
        file
    )

In [57]:
# GRU

gru_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    GRU(
        64
    ),
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [58]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [59]:
gru_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,599,361 (9.92 MB)

 Trainable params: 2,599,361 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

In [60]:
gru_history = gru_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 108s 190ms/step - accuracy: 0.5065 - loss: 0.6924 - val_accuracy: 0.5236 - val_loss: 0.6868
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 114s 211ms/step - accuracy: 0.6190 - loss: 0.5953 - val_accuracy: 0.8587 - val_loss: 0.3464
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 116s 213ms/step - accuracy: 0.9012 - loss: 0.2538 - val_accuracy: 0.8818 - val_loss: 0.2804
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 147s 223ms/step - accuracy: 0.9562 - loss: 0.1305 - val_accuracy: 0.8883 - val_loss: 0.3065
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 113s 208ms/step - accuracy: 0.9814 - loss: 0.0617 - val_accuracy: 0.8781 - val_loss: 0.3924
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 111s 205ms/step - accuracy: 0.9912 - loss: 0.0300 - val_accuracy: 0.8781 - val_loss: 0.5071
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 114s 210ms/step - accuracy: 0.9946 - loss: 0.0198 - val_accuracy: 0.8754 - val_loss: 0.5894
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 112s 206ms/step - accuracy: 0.9965 -

In [61]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step


In [62]:
gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


In [63]:
gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

In [64]:
gru_metrics

{'Accuracy': 0.872143049206776,
 'Precision': 0.8689655172413793,
 'Recall': 0.8775783552102866,
 'F1 Score': 0.8732506997201119,
 'ROC-AUC': 0.9352575219085855}

In [65]:
gru_model.save(
    "../models/gru/gru_baseline.keras"
)

In [66]:
with open(
    "../models/gru/gru_history.pkl",
    "wb"
) as file:

    pickle.dump(
        gru_history.history,
        file
    )

In [67]:
# Bidirectional LSTM

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(64)
    ),
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [68]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [69]:
bi_lstm_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,662,977 (10.16 MB)

 Trainable params: 2,662,977 (10.16 MB)

 Non-trainable params: 0 (0.00 B)

In [70]:
bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 207s 372ms/step - accuracy: 0.8339 - loss: 0.3768 - val_accuracy: 0.8876 - val_loss: 0.2847
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 170s 312ms/step - accuracy: 0.9252 - loss: 0.2039 - val_accuracy: 0.8853 - val_loss: 0.2894
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 181s 334ms/step - accuracy: 0.9540 - loss: 0.1281 - val_accuracy: 0.8836 - val_loss: 0.3459
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 184s 340ms/step - accuracy: 0.9648 - loss: 0.1020 - val_accuracy: 0.8735 - val_loss: 0.3563
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 218s 370ms/step - accuracy: 0.9789 - loss: 0.0653 - val_accuracy: 0.8712 - val_loss: 0.4459
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 185s 341ms/step - accuracy: 0.9869 - loss: 0.0413 - val_accuracy: 0.8727 - val_loss: 0.5683
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 185s 340ms/step - accuracy: 0.9899 - loss: 0.0326 - val_accuracy: 0.8688 - val_loss: 0.5442
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 189s 347ms/step - accuracy: 0.9918 -

In [71]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step


In [72]:
bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

In [73]:
bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

In [74]:
bi_lstm_metrics

{'Accuracy': 0.8643452541005646,
 'Precision': 0.8661290322580645,
 'Recall': 0.8631127779266006,
 'F1 Score': 0.8646182745203274,
 'ROC-AUC': 0.9358436789288228}

In [75]:
bi_lstm_model.save(
    "../models/bi_lstm/bi_lstm_baseline.keras"
)

In [76]:
with open(
    "../models/bi_lstm/bi_lstm_history.pkl",
    "wb"
) as file:

    pickle.dump(
        bi_lstm_history.history,
        file
    )

In [77]:
baseline_results = pd.DataFrame(
    [
        ann_metrics,
        rnn_metrics,
        lstm_metrics,
        gru_metrics,
        bi_lstm_metrics
    ],
    index=[
        "ANN",
        "RNN",
        "LSTM",
        "GRU",
        "Bi-LSTM"
    ]
)

In [78]:
baseline_results

,Accuracy,Precision,Recall,F1 Score,ROC-AUC
ANN,0.874429,0.876513,0.872756,0.874631,0.948354
RNN,0.497714,0.499779,0.909456,0.645069,0.500022
LSTM,0.865286,0.879622,0.847576,0.863302,0.932051
GRU,0.872143,0.868966,0.877578,0.873251,0.935258
Bi-LSTM,0.864345,0.866129,0.863113,0.864618,0.935844
